# LAB 0 - VM Setup
**Importing and using the course virtual machine**


Authors:
    
- Prof. Marco A. Deriu (marco.deriu@polito.it)
- Eric A. Zizzi (eric.zizzi@polito.it)
- Marcello Miceli (marcello.miceli@polito.it)

# Table of Contents

1. What you need
2. Download and verify the VM
3. Import the VM into VirtualBox
4. First start
5. Check the installation
6. Measure your simulation speed
7. Everyday use
8. Troubleshooting

**Learning outcomes:**
- import the course virtual machine (VM) into VirtualBox
- give it a sensible share of your computer's resources
- check that GROMACS, Python and VMD work inside it
- measure how fast GROMACS runs on your computer
- use the course commands to get, restore and back up your lab files

# The course VM

Every lab runs inside a **virtual machine (VM)**: a complete Linux computer that runs as an application on your own computer. It comes with everything the course needs already installed: GROMACS, VMD, Python with JupyterLab, and the course commands. The only thing you install on your own computer is VirtualBox.

Your computer is the **host** and the VM is the **guest**. While it runs, the guest borrows part of the host's processors (CPUs) and memory (RAM).

# 1. What you need

- **Hardware virtualization** turned on: Intel VT-x or AMD-V on PCs; always available on Apple Silicon Macs. See Troubleshooting if VirtualBox complains about it.
- **At least 8 GB of RAM** and **40 GB of free disk space**.
- **Oracle VirtualBox 7.2** or newer, from [virtualbox.org](https://www.virtualbox.org/wiki/Downloads). Install it like any other application.
- An **internet connection** the first time you start the VM.

## Which VM file is yours?

There are two versions of the VM. Download only the one that matches your computer:

| Your computer | VM file |
|---|---|
| Windows or Linux PC | `molbiomech-vm-<version>-amd64.ova` |
| Mac with an Intel processor | `molbiomech-vm-<version>-amd64.ova` |
| Mac with Apple Silicon (M1 or newer) | `molbiomech-vm-<version>-arm64.ova` |

On a Mac, open the Apple menu → **About This Mac**: "Chip: Apple M…" means Apple Silicon, "Processor: Intel" means Intel.

# 2. Download and verify the VM

Download the three files from the link your instructor gives you, and keep them in the same folder:

| File | What it is |
|---|---|
| `molbiomech-vm-<version>-<arch>.ova` | The VM itself (several GB) |
| `SHA256SUMS` | Checksums to verify the download |
| `manifest.json` | Version information |

## Verify the download

A large download can arrive damaged. Compute the checksum of the `.ova` file and compare it with the line for the same file in `SHA256SUMS` (open it with any text editor). The two must be identical, ignoring upper and lower case.

**Windows** (PowerShell, in the download folder):
```powershell
Get-FileHash .\molbiomech-vm-<version>-amd64.ova -Algorithm SHA256
```

**macOS** (Terminal, in the download folder):
```bash
shasum -a 256 molbiomech-vm-<version>-arm64.ova
```

**Linux** (terminal, in the download folder), which compares for you and prints `OK`:
```bash
sha256sum -c SHA256SUMS --ignore-missing
```

If the checksums differ, download the `.ova` again.

# 3. Import the VM into VirtualBox

1. Open VirtualBox and choose **File → Import Appliance…**
2. Select the `.ova` file and click **Next**.
3. In the settings list, check the **CPU** and **RAM** values and change them if needed (next slide).
4. Click **Finish**. Importing takes a few minutes.

The VM then appears in VirtualBox's list on the left, named `molbiomech-vm-…`.

## How many CPUs and how much memory?

The VM borrows CPUs and memory from your computer while it runs. **Never give it more than half of what your computer has**, or your computer itself becomes slow and unstable.

| Your computer | CPUs | Memory (RAM) |
|---|---|---|
| 8 or more logical processors, 16 GB RAM or more | 4 | 4096–8192 MB |
| 8 or more logical processors, 8 GB RAM | 4 | 4096 MB |
| Fewer than 8 logical processors | half of them | 4096 MB |

4096 MB (4 GB) is enough for every lab. More CPUs make simulations faster; more memory does not.

To count your logical processors: on **Windows**, Task Manager → Performance → CPU → "Logical processors"; on **macOS**, run `sysctl -n hw.logicalcpu` in Terminal; on **Linux**, run `nproc`.

You can change these values later: shut the VM down, then open **Settings → System**.

# 4. First start

Select the VM and click **Start**. Debian Linux boots and logs you in automatically as the user `student` (password `student`, in case you are ever asked).

On the first start, a window downloads the course material. **Keep the VM connected to the internet and wait until it finishes.** Your lab files end up in `~/molbiomech/labs`, one folder per lab.

If the VM was offline, connect it and double-click **Fetch course materials** on the desktop, or run `course-fetch` in a terminal.

## Keyboard and screen

The VM uses an **Italian keyboard layout**. If your keyboard is different, open **Applications → Settings → Keyboard → Layout**, untick *Use system defaults*, add your layout and move it to the top of the list.

If the desktop is too small or too large, change the scaling from VirtualBox's **View → Virtual Screen 1** menu.

# 5. Check the installation

Inside the VM, double-click **JupyterLab** on the desktop. JupyterLab opens in your labs folder: open `00-VMSetup/00-VMSetup.ipynb` (this notebook) and run the cells below with **Shift+Enter**.

First, check which version of the course material you have:

In [ ]:
!course-version

`course-doctor` checks your lab files, GROMACS, the Python packages and VMD. Every line should start with **PASS**. If a line says **FAIL**, or the VMD line says **SKIP**, tell your instructor.

In [ ]:
!course-doctor

GROMACS runs every simulation from LAB 3 on. Check its version:

In [ ]:
!gmx --version 2>&1 | grep -E "GROMACS version|Precision"

# 6. Measure your simulation speed

Every simulation in LABs 3–6 is sized to how fast GROMACS runs on **your** computer, measured in **ns/day**: nanoseconds of simulated time per day of computing.

Measure it now on the course molecule, CLN025 in water (3,630 atoms), with the settings used from LAB 3 on. The next cell minimises the energy of the system, then runs 40 ps of dynamics and times the last 30 ps. It takes a few minutes; wait until the output appears.

In [ ]:
%%bash
mkdir -p benchmark && cd benchmark
cp ../../common/structures/cln025_solvated/* .
gmx grompp -f ../../common/mdp/em.mdp -c cln025_solvated.gro -p topol.top -o em.tpr > grompp_em.log 2>&1
gmx mdrun -deffnm em -ntmpi 1 > mdrun_em.log 2>&1
gmx grompp -f ../../common/mdp/nvt.mdp -c em.gro -r em.gro -p topol.top -o bench.tpr > grompp_bench.log 2>&1
gmx mdrun -deffnm bench -ntmpi 1 -nsteps 20000 -resetstep 5000 -notunepme > mdrun_bench.log 2>&1
grep -B1 "Performance:" bench.log

The first number after **Performance:** is your speed in ns/day. **Write it down**: each later lab tells you how long its runs take at your speed. For example, at 20 ns/day a 10 ns simulation takes 12 hours.

If nothing is printed, look at the `.log` files in `00-VMSetup/benchmark/` and tell your instructor.

# 7. Everyday use

| Desktop icon | What it does |
|---|---|
| **JupyterLab** | Opens JupyterLab in your labs folder (same as `course-jupyter`) |
| **Course Terminal** | Opens a terminal in `~/molbiomech/labs` |
| **Course Files** | Opens the course folder in the file manager |
| **Fetch course materials** | Runs `course-fetch` with a progress window |

VMD is in the **Applications** menu.

## Course commands

Run these in a terminal:

| Command | What it does |
|---|---|
| `course-fetch` | Downloads the latest course files and the course's reference datasets. Course files you already have are never overwritten. |
| `course-fetch <dataset>` | Downloads only that reference dataset, plus any new course files; each lab tells you which dataset it needs |
| `course-refresh` | Copies back any course file missing from your labs folder, e.g. one you deleted by mistake. Works offline; your edited files are kept. |
| `course-reset <lab>` | Replaces one lab folder, e.g. `course-reset 03-ClassicalMD`, with a fresh copy of the course files as last downloaded by `course-fetch`. Your version is saved in `~/molbiomech/backups` first. `course-reset --all` does this for every lab. |
| `course-backup` | Saves your labs to a `.tar.gz` archive in `~/molbiomech/backups` and prints its name |
| `course-jupyter` | Starts JupyterLab in your labs folder (same as the desktop icon) |
| `course-doctor` | Checks the installation |
| `course-version` | Shows which version of the course material you have |

## Keep your work safe

- **Shut down properly**: inside the VM, **Applications → Log Out → Shut Down**. Closing the VirtualBox window and choosing *Power off the machine* is like pulling the plug, and unsaved work is lost.
- **Take a VirtualBox snapshot** once everything works: while the VM is running, **Machine → Take Snapshot…**. If the VM ever breaks, restore the snapshot from VirtualBox's Snapshots view, or import the `.ova` again. Both discard everything created afterwards, including the archives made by `course-backup`, which are stored inside the VM.

# 8. Troubleshooting

| Problem | What to do |
|---|---|
| VirtualBox says hardware virtualization (VT-x/AMD-V) is not available | Turn on virtualization (Intel VT-x, or AMD SVM/AMD-V) in your computer's BIOS/UEFI settings, then try again |
| Your computer becomes very slow while the VM runs | Shut the VM down and lower its CPUs and memory (section 3) |
| The keyboard types the wrong characters | Set your keyboard layout (section 4) |
| `~/molbiomech/labs` is empty | Connect to the internet and run `course-fetch`, or double-click **Fetch course materials** |
| A green turtle icon appears in VirtualBox's status bar (Windows) | The VM is running through Windows' Hyper-V and will be slower. Tell your instructor; don't change Windows security settings on your own. |